# Statistical Analysis

We will use two business questions and compare receipt values:

1. Do receipts with a promotion have a different average basket value from receipts without a promotion?
2. Do receipts paid with M-Pesa have a different average net value from receipts paid with cash?

## 1. Load the cleaned data

Run the data_preparation notebook first. This notebook reads its cleaned CSV. It uses SciPy for Welch's t-test and its 95% confidence interval.

In [2]:
from pathlib import Path
import pandas as pd
from scipy import stats
from IPython.display import display

ROOT = Path.cwd()
while not (ROOT / 'data' / 'processed' / 'clean_sales_2025.csv').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

SALES_FILE = ROOT / 'data' / 'processed' / 'clean_sales_2025.csv'
if not SALES_FILE.exists():
    raise FileNotFoundError('Run the data_preparation notebook first to create the cleaned sales file.')

sales = pd.read_csv(SALES_FILE, parse_dates=['sale_date'])
REPORTS_DIR = ROOT / 'reports'
print(f'Cleaned product lines: {len(sales):,}')
print(f'Dates: {sales.sale_date.min():%d %b %Y} to {sales.sale_date.max():%d %b %Y}')

Cleaned product lines: 1,511
Dates: 01 Jan 2025 to 31 Dec 2025


## 2. Make one row for each receipt

The cleaned file has one row per product on a receipt. We add those rows together so each receipt is counted once. A receipt is marked as promoted if at least one item had a discount. For the promotion question, we compare gross sales before discounts so the discount itself does not automatically lower the promoted group's result.

In [3]:
sales['has_promotion_line'] = sales['promotion_pct'].gt(0)
receipts = sales.groupby('transaction_id', as_index=False).agg(
    sale_date=('sale_date', 'min'),
    customer_id=('customer_id', 'first'),
    payment_method=('payment_method', 'first'),
    gross_receipt_kes=('gross_sales_kes', 'sum'),
    net_receipt_kes=('net_sales_kes', 'sum'),
    has_promotion=('has_promotion_line', 'max'),
)
assert receipts['transaction_id'].is_unique
print(f'Receipts to analyse: {len(receipts):,}')

promotion_check = receipts.groupby('has_promotion')['gross_receipt_kes'].describe().round(2)
print('Gross receipt values: no promotion (False) and at least one promotion (True)')
display(promotion_check)

payment_receipts = receipts[receipts['payment_method'].isin(['M-Pesa', 'Cash'])].copy()
payment_check = payment_receipts.groupby('payment_method')['net_receipt_kes'].describe().round(2)
print('Net receipt values: M-Pesa and cash (card receipts are excluded)')
display(payment_check)

Receipts to analyse: 1,134
Gross receipt values: no promotion (False) and at least one promotion (True)


,count,mean,std,min,25%,50%,75%,max
has_promotion,,,,,,,,
False,857.0,322.54,305.46,60.0,140.0,225.0,400.0,2240.0
True,277.0,373.14,289.45,60.0,170.0,320.0,500.0,2240.0


Net receipt values: M-Pesa and cash (card receipts are excluded)


,count,mean,std,min,25%,50%,75%,max
payment_method,,,,,,,,
Cash,397.0,333.55,313.78,57.0,142.5,225.00,417.00,2240.0
M-Pesa,558.0,337.75,294.00,57.0,145.0,251.25,455.75,2048.0


## 3. Test 1: promotion and receipt value

**Question:** Is the average gross receipt value different when a receipt includes a promotion?

- Null hypothesis: the two groups have the same average gross receipt value.
- Alternative hypothesis: their average values are different.

We use a two-sided Welch t-test and a 95% confidence interval for the difference (promoted minus not promoted). This checks for an association. Promotions were not randomly assigned, so a difference does not prove that promotions caused a change in sales.

In [4]:
promoted = receipts.loc[receipts['has_promotion'], 'gross_receipt_kes']
not_promoted = receipts.loc[~receipts['has_promotion'], 'gross_receipt_kes']
promotion_test = stats.ttest_ind(promoted, not_promoted, equal_var=False)
promotion_ci = promotion_test.confidence_interval(confidence_level=0.95)
promotion_difference = promoted.mean() - not_promoted.mean()

print(f'Promoted receipts: {len(promoted):,}; average gross value: KES {promoted.mean():,.2f}')
print(f'Non-promoted receipts: {len(not_promoted):,}; average gross value: KES {not_promoted.mean():,.2f}')
print(f'Mean difference: KES {promotion_difference:,.2f}')
print(f'95% confidence interval: KES {promotion_ci.low:,.2f} to KES {promotion_ci.high:,.2f}')
print(f'Welch t-test p-value: {promotion_test.pvalue:.4g}')

if promotion_test.pvalue < 0.05:
    if promotion_difference > 0:
        promotion_conclusion = f'Promoted receipts had a higher average gross value by KES {promotion_difference:,.2f}. The difference is statistically significant in this sample, but it does not show that the promotion caused the higher value.'
    else:
        promotion_conclusion = f'Promoted receipts had a lower average gross value by KES {abs(promotion_difference):,.2f}. The difference is statistically significant in this sample, but it does not show that the promotion caused the lower value.'
else:
    promotion_conclusion = f'The data does not provide clear evidence of a difference in average gross receipt value. The observed difference is KES {promotion_difference:,.2f}; the 95% interval includes no difference.'
print('Plain-language conclusion:', promotion_conclusion)

Promoted receipts: 277; average gross value: KES 373.14
Non-promoted receipts: 857; average gross value: KES 322.54
Mean difference: KES 50.60
95% confidence interval: KES 10.75 to KES 90.45
Welch t-test p-value: 0.01293
Plain-language conclusion: Promoted receipts had a higher average gross value by KES 50.60. The difference is statistically significant in this sample, but it does not show that the promotion caused the higher value.


## 4. Test 2: M-Pesa and cash receipt value

**Question:** Is the average amount paid different for M-Pesa receipts and cash receipts?

- Null hypothesis: the two payment groups have the same average net receipt value.
- Alternative hypothesis: their average values are different.

We compare net sales because it is the amount after discounts. This is a receipt-level comparison, not a claim that individual M-Pesa customers spend more. Card receipts are left out of this two-group test.

In [ ]:
mpesa = payment_receipts.loc[payment_receipts['payment_method'] == 'M-Pesa', 'net_receipt_kes']
cash = payment_receipts.loc[payment_receipts['payment_method'] == 'Cash', 'net_receipt_kes']
payment_test = stats.ttest_ind(mpesa, cash, equal_var=False)
payment_ci = payment_test.confidence_interval(confidence_level=0.95)
payment_difference = mpesa.mean() - cash.mean()

print(f'M-Pesa receipts: {len(mpesa):,}; average net value: KES {mpesa.mean():,.2f}')
print(f'Cash receipts: {len(cash):,}; average net value: KES {cash.mean():,.2f}')
print(f'Mean difference (M-Pesa minus cash): KES {payment_difference:,.2f}')
print(f'95% confidence interval: KES {payment_ci.low:,.2f} to KES {payment_ci.high:,.2f}')
print(f'Welch t-test p-value: {payment_test.pvalue:.4g}')

if payment_test.pvalue < 0.05:
    if payment_difference > 0:
        payment_conclusion = f'M-Pesa receipts had a higher average net value by KES {payment_difference:,.2f}. The difference is statistically significant in this sample.'
    else:
        payment_conclusion = f'M-Pesa receipts had a lower average net value by KES {abs(payment_difference):,.2f}. The difference is statistically significant in this sample.'
else:
    payment_conclusion = f'The data does not provide clear evidence of a difference in average net receipt value. The observed difference is KES {payment_difference:,.2f}; the 95% interval includes no difference.'
print('Conclusion:', payment_conclusion)

M-Pesa receipts: 558; average net value: KES 337.75
Cash receipts: 397; average net value: KES 333.55
Mean difference (M-Pesa minus cash): KES 4.20
95% confidence interval: KES -35.20 to KES 43.60
Welch t-test p-value: 0.8345
Plain-language conclusion: The data does not provide clear evidence of a difference in average net receipt value. The observed difference is KES 4.20; the 95% interval includes no difference.


## 5. Assumptions and limits

- The unit is one receipt, and each receipt appears once in the comparison.
- The tables above show sample sizes, means, medians, spread, and extremes. A few large baskets can pull up the mean; we keep them because a large basket can be a real sale.
- Each group has more than 30 receipts. Welch's test allows the groups to have different variances.
- The same customer ID may appear on multiple receipts, so independence across receipts is not guaranteed. Treat the p-values as exploratory.
- The dataset is synthetic and promotions were not randomly assigned. These tests cannot establish cause and effect or represent actual Nairobi customers.

In [ ]:
# Check assumptions
promotion_min_n = min(len(promoted), len(not_promoted))
payment_min_n = min(len(mpesa), len(cash))
groups_large_enough = promotion_min_n > 30 and payment_min_n > 30

In [11]:
assumption_note = (
    'Each observation is one receipt and each receipt appears once. The group summaries show group sizes, means, medians, spreads, and extremes. '
    f'The smallest promotion/payment group sizes are {promotion_min_n} and {payment_min_n}; both exceed 30: {groups_large_enough}. '
    'Welch t-tests allow unequal group variances. '
    'Some customer IDs appear on multiple receipts, so independence across receipts is not guaranteed. '
    'The generated data and non-random promotion assignment mean these are exploratory associations, not causal estimates.'
)

report_lines = [
    '# Statistical Analysis Results',
    '',
    'These tests use synthetic receipt data for 2025. Differences describe this sample and do not prove cause and effect.',
    '',
    '## Assumptions checked',
    '',
    assumption_note,
    '',
    '## Test 1: promotion and gross receipt value',
    '',
    f'Receipts with promotion: {len(promoted)}; mean KES {promoted.mean():.2f}.',
    f'Receipts without promotion: {len(not_promoted)}; mean KES {not_promoted.mean():.2f}.',
    f'Mean difference (with minus without): KES {promotion_difference:.2f}.',
    f'95% confidence interval: KES {promotion_ci.low:.2f} to KES {promotion_ci.high:.2f}.',
    f'Welch t-test p-value: {promotion_test.pvalue:.4g}.',
    f'Conclusion: {promotion_conclusion}',
    '',
    '## Test 2: M-Pesa and cash net receipt value',
    '',
    f'M-Pesa receipts: {len(mpesa)}; mean KES {mpesa.mean():.2f}.',
    f'Cash receipts: {len(cash)}; mean KES {cash.mean():.2f}.',
    f'Mean difference (M-Pesa minus cash): KES {payment_difference:.2f}.',
    f'95% confidence interval: KES {payment_ci.low:.2f} to KES {payment_ci.high:.2f}.',
    f'Welch t-test p-value: {payment_test.pvalue:.4g}.',
    f'Conclusion: {payment_conclusion}',
]

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_FILE = REPORTS_DIR / 'results.md'
RESULTS_FILE.write_text('\n'.join(report_lines) + '\n', encoding='utf-8')
print(f'Results saved to {RESULTS_FILE.relative_to(ROOT)}')

Results saved to reports\results.md
